# Healthcare access

Motorized land travel time (minutes) to the nearest hospital or clinic, from **[Weiss et al. 2020 — Global maps of travel time to healthcare facilities](https://malariaatlas.org/project-resources/accessibility-to-healthcare/)** (MAP / University of Oxford, Telethon Kids, Google, University of Twente). The source raster is a 30 arc-second (~1 km) global GeoTIFF covering 85°N to 60°S, distributed as a zip via the Malaria Atlas Project data portal.

We aggregate ~3600 source pixels per 0.5° atlas cell as the **mean of `log10(1 + minutes)`** — travel time spans four orders of magnitude between city centres (~1 min) and remote wilderness (~10 000 min), so an arithmetic mean lets a handful of unreachable pixels dominate a cell that's otherwise well-served. The log transform is standard for skewed access measures and mirrors what `internet_connectivity` does with download speed.

Sign is inverted so higher = shorter travel time = better access. Cells with no source coverage (Antarctica, high Arctic) stay `NaN`; ocean is masked via `is_land` from `grid.nc`. `weighted_score` renormalizes weights per-cell, so unknown healthcare access neither drags nor lifts a location. Motorized is the default because livability implicitly assumes access to some form of transport; swap `PRODUCT = 'walking_only'` to score the pedestrian scenario instead.

In [1]:
import zipfile

import numpy as np
import rasterio
import xarray as xr
from rasterio.windows import Window
from tqdm.auto import tqdm

from common import RAW_DIR, download, load_grid, plot_map, save_variable

VARIABLE = 'healthcare_access'
variable_raw = RAW_DIR / VARIABLE
variable_raw.mkdir(parents=True, exist_ok=True)

# 'motorized' or 'walking_only' — MAP publishes both scenarios from the same
# 2020 study. Motorized is the canonical livability signal.
PRODUCT = 'motorized'
DATASET = f'2020_{PRODUCT}_travel_time_to_healthcare'

grid = load_grid()
NLAT, NLON = grid.lat.size, grid.lon.size
RES = float(grid.lat.diff('lat').mean())
LAT_MIN = float(grid.lat.min()) - RES / 2
LON_MIN = float(grid.lon.min()) - RES / 2
print(f'atlas grid: {NLAT} x {NLON} at {RES}°')

atlas grid: 360 x 720 at 0.5°


## 1. Fetch raw data

The MAP data portal exposes a `DirectDownload` endpoint that returns a zip containing a single GeoTIFF (~200 MB compressed, ~1 GB expanded). Cached under `variable_raw`; the download only runs once.

In [ ]:
url = 'https://data.malariaatlas.org/geoserver/ows?service=CSW&version=2.0.1&request=DirectDownload&ResourceId=Explorer:2020_motorized_travel_time_to_healthcare'
zip_path = variable_raw / f'{DATASET}.zip'
tif_path = variable_raw / f'{DATASET}.tif'

if not tif_path.exists():
    if not zip_path.exists():
        print(f'downloading {url}')
        await download(url, zip_path)
    with zipfile.ZipFile(zip_path) as z:
        # MAP zips wrap the raster in a folder; find the .tif regardless of its exact name.
        member = next(n for n in z.namelist() if n.lower().endswith(('.tif', '.tiff', '.geotiff')))
        with z.open(member) as src, open(tif_path, 'wb') as dst:
            dst.write(src.read())

print(f'{tif_path.name}: {tif_path.stat().st_size / 1024**2:.0f} MB')

downloading https://data.malariaatlas.org/geoserver/ows?service=CSW&version=2.0.1&request=DirectDownload&ResourceId=Explorer:2020_motorized_travel_time_to_healthcare


ReadTimeout: 

## 2. Aggregate source pixels into atlas cells

Same block-streaming pattern as `21_population_density.ipynb`: `row_idx` is monotonic along the source y-axis, so all source rows that fall into the same atlas row form one contiguous on-disk block. We iterate over those blocks, `log10(1 + minutes)` inside the block, then bin along x with `np.bincount`.

Two accumulators are needed for a true mean: `log_sum` (Σ log₁₀(1 + t) over valid pixels) and `counts` (number of valid pixels). Dividing element-wise at the end gives the per-cell mean; cells with `counts == 0` stay `NaN`. NoData sentinels (very large sentinel ints in MAP rasters) and negative values are filtered before the sum.

In [ ]:
log_sum = np.zeros((NLAT, NLON), dtype='float64')
counts = np.zeros((NLAT, NLON), dtype='float64')

with rasterio.open(tif_path) as src:
    assert src.crs.to_epsg() == 4326, f'expected EPSG:4326, got {src.crs}'
    nodata = src.nodata

    src_lon = src.transform.c + src.transform.a * (np.arange(src.width) + 0.5)
    src_lat = src.transform.f + src.transform.e * (np.arange(src.height) + 0.5)
    col_idx = np.floor((src_lon - LON_MIN) / RES).astype(np.int64)
    row_idx = np.floor((src_lat - LAT_MIN) / RES).astype(np.int64)
    col_valid = (col_idx >= 0) & (col_idx < NLON)
    valid_cols = col_idx[col_valid]

    change_at = np.flatnonzero(np.diff(row_idx)) + 1
    starts = np.concatenate([[0], change_at])
    ends = np.concatenate([change_at, [src.height]])

    for s, e in tqdm(list(zip(starts, ends)), desc='atlas rows'):
        arow = int(row_idx[s])
        if arow < 0 or arow >= NLAT:
            continue
        block = src.read(1, window=Window(0, int(s), src.width, int(e - s))).astype('float64')
        # Weiss et al. rasters use a large sentinel for unreachable/nodata cells; also drop
        # negatives and NaNs defensively so log10 stays defined.
        mask = np.isfinite(block) & (block >= 0)
        if nodata is not None:
            mask &= (block != nodata)
        log_block = np.where(mask, np.log10(1.0 + block), 0.0)
        row_log_sum = log_block.sum(axis=0)
        row_count = mask.sum(axis=0).astype('float64')
        log_sum[arow] += np.bincount(valid_cols, weights=row_log_sum[col_valid], minlength=NLON)[:NLON]
        counts[arow] += np.bincount(valid_cols, weights=row_count[col_valid], minlength=NLON)[:NLON]

with np.errstate(invalid='ignore', divide='ignore'):
    log_mean_time = np.where(counts > 0, log_sum / counts, np.nan)

coverage = float(np.isfinite(log_mean_time).sum()) / (NLAT * NLON)
print(f'{coverage * 100:.1f}% of atlas cells have Weiss coverage')

## 3. Sign-invert and mask ocean

Sign is flipped so higher = shorter travel time = better access, matching the atlas convention. Ocean cells (and the Antarctica tail below 60°S that Weiss et al. don't cover) are masked via `is_land`.

In [ ]:
values = xr.DataArray(
    (-log_mean_time).astype('float32'),
    coords={'lat': grid.lat, 'lon': grid.lon},
    dims=('lat', 'lon'),
    name=VARIABLE,
).where(grid.is_land == 1)
values.attrs['units'] = '-log₁₀(1 + minutes)'
values.attrs['source'] = f'Weiss et al. 2020 ({DATASET})'
values.attrs['aggregation'] = 'mean of log10(1 + travel_time_min) per 0.5° cell, sign-inverted'

finite = values.where(np.isfinite(values))
print(f'range: {float(finite.min()):.2f} .. {float(finite.max()):.2f} '
      f'(i.e. {10 ** -float(finite.max()) - 1:.0f} .. {10 ** -float(finite.min()) - 1:.0f} min)')

## 4. Plot

In [ ]:
plot_map(values, cmap='RdYlGn', robust=True)

## 5. Save

In [ ]:
out = save_variable(values, VARIABLE)
print(f'wrote {out}')